# PySpark partitionBy() 
is a function used to partition the large dataset (DataFrame) into smaller files based on one or multiple columns while writing to disk

- PySpark supports partition in two ways; partition in memory (DataFrame) and partition on the disk (File system).
- Partition in memory: You can partition or repartition the DataFrame by calling repartition() or coalesce() transformations.
- Partition on disk: While writing the PySpark DataFrame back to disk, you can choose how to partition the data based on columns using partitionBy() of pyspark.sql.DataFrameWriter.

df.write.option("header",True) \
  .partitionBy("state") \
  .mode("overwrite") \
  .csv("/tmp/zipcodes-state")

dfRepart.repartition(2) \
    .write.option("header",True) \
    .partitionBy("state") \
    .mode("overwrite") \
    .csv("c:/tmp/zipcodes-state-more")


- partitionBy in PySpark is used in two different contexts depending on whether you are talking about:

- Writing DataFrames to disk (partitioned files)

- Window functions (partitioning rows for calculations)

1. partitionBy in DataFrameWriter (when saving data)

When you write a DataFrame to a file format (like Parquet, ORC, CSV), partitionBy organizes the output into separate subdirectories based on column values.

In [0]:
df.write.partitionBy("country").parquet("/tmp/output")


If your DataFrame has a country column with values USA and India, Spark will create:

In [0]:
/tmp/output/country=USA/...
/tmp/output/country=India/...


This is very useful for big data queries (partition pruning in Hive, Presto, Spark SQL).

👉 You can also partition by multiple columns:

In [0]:
df.write.partitionBy("country", "year").parquet("/tmp/output")


2. partitionBy in Window functions

When used with Window (from pyspark.sql.window), partitionBy groups rows into partitions (like SQL PARTITION BY) so that window functions (row_number, rank, sum, etc.) operate within each group.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("country").orderBy("salary")
df.withColumn("row_num", row_number().over(window)).show()

# If country = USA has 5 rows, they get numbered 1 to 5 independently from rows where country = India.

✅ Summary

df.write.partitionBy(...) → splits output files into subdirectories by column values.

Window.partitionBy(...) → splits rows into groups for window calculations.